# Libera L1B - Observation Geometry Fields

The L1B radiometer product carries the observation geometry alongside the radiances: where the
boresight hit the ground, where the spacecraft and Sun were, and the angles relating them.
Downstream products depend on these - footprint matching against the WFOV camera, angular
distribution models, and flux inversion all key off them - so it is worth knowing what each field
means and what a correct one looks like.

This notebook has two halves:

1. **The product.** Open a bundled L1B granule, inventory the geometry fields, plot them, and run a
   set of internal-consistency checks that must hold if the fields agree with each other.
2. **The source.** Reproduce the same fields from SPICE with `libera_rad.geolocation`, confirm they
   match the product to float32 rounding, and compute them on a different time grid.

Everything runs on data in this repository:

| What | Where |
| --- | --- |
| L1B granule (30 s, 100 Hz, 3000 samples) | `learning_notebooks/sample_data/LIBERA_L1B_RAD-4CH_V0-6-1_*.nc` |
| SPICE kernels it was produced from | `tests/test_data/l1b_integration_data/` (AZROT-CK, ELSCAN-CK, JPSS-CK, JPSS-SPK) |

The granule is a cross-track scan: the azimuth motor is parked and the elevation motor sweeps the
boresight through about six full cycles - twelve limb-to-limb traversals - in 30 seconds.

In [ ]:
import logging
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


def repo_root(start: Path | None = None) -> Path:
    """Walk up from `start` until the directory containing pyproject.toml is found."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from " + str(here))


ROOT = repo_root()
SAMPLE_DIR = ROOT / "learning_notebooks" / "sample_data"
KERNEL_DIR = ROOT / "tests" / "test_data" / "l1b_integration_data"

logging.getLogger().setLevel(logging.WARNING)

# Samples whose boresight misses the ellipsoid are expected, and curryer returns NaN for them via an
# invalid sqrt. Silencing that keeps the off-Earth samples from burying every cell in warnings.
np.seterr(invalid="ignore")

xr.set_options(display_width=120, display_max_rows=40)
np.set_printoptions(linewidth=120, suppress=True)
pd.set_option("display.width", 160, "display.max_rows", 80, "display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (12, 4.5), "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

print("repository root:", ROOT)

## 1. The product as delivered

Open the granule the way a downstream consumer would. `mask_and_scale=True` (the xarray default)
turns the product `_FillValue` entries into NaN, which is what we want for plotting and for
reasoning about coverage.

In [ ]:
l1b_path = sorted(SAMPLE_DIR.glob("LIBERA_L1B_RAD-4CH_V0-6-1_*.nc"))[-1]
ds = xr.open_dataset(l1b_path).load()

time = ds["radiometer_time"].values
seconds = (time - time[0]) / np.timedelta64(1, "s")

print("file          :", l1b_path.name)
print("dimensions    :", dict(ds.sizes))
print("time span     :", time[0], "->", time[-1])
print("cadence       : %.1f Hz over %.2f s" % (1.0 / np.median(np.diff(seconds)), seconds[-1]))
print("algorithm ver :", ds.attrs.get("algorithm_version"))
print("earth-sun AU  : %.6f" % float(ds.attrs["Earth_Sun_Distance_AU"]))
print("variables     :", len(ds.data_vars))

### 1.1 The geometry fields, grouped

Five families. The distinction that matters most when reading the product: the **surface** fields
describe the point the boresight hit and are NaN whenever it missed the Earth, whereas the
**spacecraft** fields are defined at every sample regardless of where the instrument pointed.

The table is built from the file's own attributes, so it cannot drift from the product definition.
`% finite` is the fraction of samples that are neither fill nor NaN.

In [ ]:
FIELD_GROUPS = {
    "boresight surface point": {
        "Latitude": "geodetic latitude of the boresight ellipsoid intersection (WGS84)",
        "Longitude": "geodetic longitude of the same point",
        "Colatitude": "90 - Latitude",
        "Altitude": "height of the intersection above the ellipsoid (0 by construction)",
        "Terrain_Corrected_Latitude": "not yet implemented - fill",
        "Terrain_Corrected_Longitude": "not yet implemented - fill",
        "Terrain_Corrected_Altitude": "not yet implemented - fill",
    },
    "reference ground points": {
        "Subsatellite_Latitude": "ground point directly beneath the spacecraft",
        "Subsatellite_Longitude": "ground point directly beneath the spacecraft",
        "Subsatellite_Colatitude": "90 - Subsatellite_Latitude",
        "Subsolar_Latitude": "ground point directly beneath the Sun",
        "Subsolar_Longitude": "ground point directly beneath the Sun",
        "Subsolar_Colatitude": "90 - Subsolar_Latitude",
    },
    "viewing and illumination": {
        "Viewing_Zenith_Surface": "zenith angle of the spacecraft seen from the surface point",
        "Solar_Zenith_Surface": "zenith angle of the Sun at the surface point",
        "Viewing_Azimuth_Surface_WRT_North": "azimuth to the spacecraft, clockwise from geodetic North",
        "Solar_Azimuth_Surface_WRT_North": "azimuth to the Sun, clockwise from geodetic North",
        "Relative_Azimuth_Surface": "mod(viewing - solar + 180, 360); the Sun sits at 180",
    },
    "scan and pointing": {
        "Cone_Angle": "boresight angle off the spacecraft-to-geocenter (nadir) vector",
        "Cone_Angle_Rate": "time derivative of Cone_Angle",
        "Clock_Angle": "boresight azimuth about nadir in the inertial-velocity frame (CERES SCI-12)",
        "Clock_Angle_Rate": "time derivative of Clock_Angle; filled near nadir where it is singular",
        "Along_Track_Angle": "boresight look angle from nadir in the velocity-nadir plane",
        "Cross_Track_Angle": "boresight look angle from nadir in the cross-track-nadir plane",
        "Line_Of_Sight": "boresight unit vector in J2000",
        "Azimuth": "azimuth motor encoder angle (SPICE CK Euler angle, not a curryer field)",
        "Elevation": "elevation motor encoder angle (SPICE CK Euler angle, not a curryer field)",
    },
    "spacecraft state": {
        "Satellite_Position": "spacecraft position in J2000",
        "Satellite_Velocity": "spacecraft velocity in J2000",
        "Satellite_Position_Start_Of_Hour": "position on a fixed 24-hour grid, not the sample grid",
        "Satellite_Velocity_Start_Of_Hour": "velocity on the same 24-hour grid",
        "Radius_of_Satellite_from_Center_of_Earth": "distance from the geocenter",
        "Satellite_Attitude_Q0": "body attitude quaternion, scalar part (Earth-fixed target frame)",
        "Satellite_Attitude_Q1": "body attitude quaternion, vector part",
        "Satellite_Attitude_Q2": "body attitude quaternion, vector part",
        "Satellite_Attitude_Q3": "body attitude quaternion, vector part",
    },
}


def inventory(dataset: xr.Dataset, groups: dict) -> pd.DataFrame:
    """Tabulate each grouped variable's declared metadata and finite fraction."""
    rows = []
    for group, members in groups.items():
        for name, description in members.items():
            values = dataset[name].values.astype("float64")
            attrs = dataset[name].attrs
            valid = attrs.get("valid_range")
            rows.append(
                {
                    "group": group,
                    "variable": name,
                    "dims": " x ".join(str(d) for d in dataset[name].dims),
                    "units": attrs.get("units", "-"),
                    "valid_range": f"[{valid[0]:g}, {valid[1]:g}]" if valid is not None else "-",
                    "% finite": round(100.0 * np.isfinite(values).mean(), 1),
                    "meaning": description,
                }
            )
    return pd.DataFrame(rows).set_index(["group", "variable"])


GEOMETRY_VARS = [name for members in FIELD_GROUPS.values() for name in members]
inventory(ds, FIELD_GROUPS)

Note the two `% finite` populations: the surface fields sit near 79% while everything spacecraft-
side is 100%. Section 1.4 shows that gap is entirely the Earth limb, not a kernel problem.

`Azimuth` and `Elevation` are worth calling out. They are motor encoder Euler angles read from the
CK frame chain, not curryer geometry fields, and they are relative to the motor frames rather than
to nadir or to the spacecraft attitude. In this granule the azimuth motor is parked near 360 and
the elevation motor does all the work.

### 1.2 Where the instrument looked

The boresight ground point against the subsatellite track. Over 30 seconds the subsatellite point
advances under 2 degrees of latitude, while the scan drags the boresight across more than 60 degrees
of longitude - the scan dominates the picture entirely, which is exactly what a cross-track granule
should look like. The printed extents below quantify both.

In [ ]:
def globe_axes(center_lon: float, center_lat: float, figsize=(9, 9)):
    """Orthographic GeoAxes centred on the given point, with coastlines and a graticule."""
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=ccrs.Orthographic(center_lon, center_lat))
    ax.set_global()
    ax.coastlines(linewidth=0.6)
    ax.gridlines(linewidth=0.3, linestyle="--", alpha=0.6)
    return fig, ax


lat, lon = ds["Latitude"].values, ds["Longitude"].values
sub_lat, sub_lon = ds["Subsatellite_Latitude"].values, ds["Subsatellite_Longitude"].values
on_earth = np.isfinite(lat)

fig, ax = globe_axes(float(np.nanmean(lon)), float(np.nanmean(lat)))
scatter = ax.scatter(
    lon[on_earth], lat[on_earth], c=seconds[on_earth], s=3, cmap="plasma",
    transform=ccrs.PlateCarree(), zorder=6, label="boresight ground point",
)
ax.plot(sub_lon, sub_lat, color="white", linewidth=2.5, transform=ccrs.PlateCarree(), zorder=8)
ax.plot(sub_lon, sub_lat, color="black", linewidth=1.2, transform=ccrs.PlateCarree(), zorder=9,
        label="subsatellite track")
ax.scatter(ds["Subsolar_Longitude"].values[0], ds["Subsolar_Latitude"].values[0], marker="*",
           s=320, color="gold", edgecolor="black", linewidth=0.6,
           transform=ccrs.PlateCarree(), zorder=10, label="subsolar point")
plt.colorbar(scatter, ax=ax, shrink=0.55, pad=0.03, label="seconds from granule start")
ax.legend(loc="upper left")
ax.set_title("Libera radiometer - boresight ground point and subsatellite track")
plt.show()

print("boresight  lat %.2f .. %.2f   lon %.2f .. %.2f" % (np.nanmin(lat), np.nanmax(lat), np.nanmin(lon), np.nanmax(lon)))
print("subsatellite  lat %.2f .. %.2f   lon %.2f .. %.2f" % (sub_lat.min(), sub_lat.max(), sub_lon.min(), sub_lon.max()))
print("off-Earth samples: %d of %d (%.1f%%)" % ((~on_earth).sum(), len(lat), 100 * (~on_earth).mean()))

### 1.3 The scan

`Cross_Track_Angle` and `Elevation` trace the same motion in different frames: the first is a look
angle from nadir, the second is the raw motor encoder. They are **sign-inverted** with respect to
each other - the two frames wind in opposite directions - so the panel below shows mirrored
sawtooths rather than overlapping ones. Section 2 checks that relation quantitatively; it is the one
place where motor telemetry and derived pointing can be compared directly.

`Along_Track_Angle` stays within about a degree of zero, which is the signature of a pure cross-track
scan. It is not identically zero, and it oscillates at twice the scan frequency: the fixed
boresight-to-motor-frame offset in the FK projects differently as the boresight swings, so the
residual along-track component peaks twice per sweep.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(seconds, ds["Cross_Track_Angle"].values, lw=0.9, label="Cross_Track_Angle")
axes[0].plot(seconds, ds["Elevation"].values, lw=0.9, ls="--", label="Elevation (motor encoder)")
axes[0].set_ylabel("degrees")
axes[0].set_title("Cross-track scan: look angle from nadir vs motor encoder")

axes[1].plot(seconds, ds["Cone_Angle"].values, lw=0.9, color="tab:green", label="Cone_Angle")
axes[1].fill_between(seconds, 0, 90, where=~on_earth, color="grey", alpha=0.25, label="boresight off Earth")
axes[1].set_ylabel("degrees")
axes[1].set_ylim(0, 90)
axes[1].set_title("Cone angle (off-nadir); shading marks samples that miss the ellipsoid")

axes[2].plot(seconds, ds["Along_Track_Angle"].values, lw=0.9, color="tab:red", label="Along_Track_Angle")
axes[2].set_ylabel("degrees")
axes[2].set_xlabel("seconds from granule start")
axes[2].set_title("Along-track angle stays near zero in a cross-track scan")

for ax in axes:
    ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

scan = ds["Cross_Track_Angle"].values
crossings = np.where(np.diff(np.signbit(np.nan_to_num(scan))))[0]
print("nadir crossings in 30 s: %d  (~%.1f s per half sweep)" % (len(crossings), seconds[-1] / max(len(crossings), 1)))
print("Along_Track_Angle range: %.3f .. %.3f deg" % (np.nanmin(ds["Along_Track_Angle"]), np.nanmax(ds["Along_Track_Angle"])))
print("Azimuth range          : %.4f .. %.4f deg" % (np.nanmin(ds["Azimuth"]), np.nanmax(ds["Azimuth"])))

### 1.4 The Earth limb decides which samples are valid

Every surface field shares one NaN mask, and that mask is set by a single geometric condition: the
boresight cone angle exceeding the limb half-angle `asin(R_earth / R_spacecraft)`. If the masks ever
disagree, or the cut-off is not at the limb, something is wrong upstream - a kernel gap would show
up as a contiguous block in time instead.

In [ ]:
SURFACE_FIELDS = [
    "Latitude", "Longitude", "Colatitude", "Altitude",
    "Viewing_Zenith_Surface", "Solar_Zenith_Surface",
    "Viewing_Azimuth_Surface_WRT_North", "Solar_Azimuth_Surface_WRT_North",
    "Relative_Azimuth_Surface",
]

reference_mask = ~np.isfinite(ds["Latitude"].values)
mask_report = pd.DataFrame(
    [
        {
            "variable": name,
            "n_nan": int(np.isnan(ds[name].values).sum()),
            "mask identical to Latitude": bool(np.array_equal(np.isnan(ds[name].values), reference_mask)),
        }
        for name in SURFACE_FIELDS
    ]
).set_index("variable")
print(mask_report.to_string(), "\n")

cone = ds["Cone_Angle"].values.astype("float64")
radius = ds["Radius_of_Satellite_from_Center_of_Earth"].values.astype("float64")
limb_angle = np.degrees(np.arcsin(6378.1366 / radius.mean()))

print("largest cone angle still on Earth : %.3f deg" % np.nanmax(cone[~reference_mask]))
print("smallest cone angle off Earth     : %.3f deg" % np.nanmin(cone[reference_mask]))
print("geometric limb asin(Re/Rsc)       : %.3f deg" % limb_angle)
print("off-Earth samples contiguous?     :", np.diff(np.where(reference_mask)[0]).max() > 1 and "no - several sweeps" or "yes")

### 1.5 Viewing and illumination angles

The four surface angles plus their derived relative azimuth. `Viewing_Zenith_Surface` tracks the
scan almost symmetrically about nadir; `Solar_Zenith_Surface` varies much less because the Sun
barely moves in 30 seconds - its variation is dominated by the surface point moving, not by the Sun.
The bottom-right panel is the diagnostic view: viewing zenith against cross-track angle should be a
clean, near-symmetric V with no hysteresis between sweeps.

In [ ]:
vza = ds["Viewing_Zenith_Surface"].values
sza = ds["Solar_Zenith_Surface"].values
vaz = ds["Viewing_Azimuth_Surface_WRT_North"].values
saz = ds["Solar_Azimuth_Surface_WRT_North"].values
raa = ds["Relative_Azimuth_Surface"].values

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].plot(seconds, vza, lw=0.9, label="Viewing_Zenith_Surface")
axes[0, 0].plot(seconds, sza, lw=0.9, label="Solar_Zenith_Surface")
axes[0, 0].set_title("Zenith angles")
axes[0, 0].set_ylabel("degrees")

axes[0, 1].plot(seconds, vaz, lw=0.9, label="Viewing_Azimuth (WRT North)")
axes[0, 1].plot(seconds, saz, lw=0.9, label="Solar_Azimuth (WRT North)")
axes[0, 1].set_title("Azimuth angles, clockwise from North")
axes[0, 1].set_ylabel("degrees")

axes[1, 0].plot(seconds, raa, lw=0.9, color="tab:purple", label="Relative_Azimuth_Surface")
axes[1, 0].axhline(180, color="gold", lw=1.5, ls="--", label="Sun direction (180)")
axes[1, 0].set_title("Relative azimuth; 180 is the solar principal plane")
axes[1, 0].set_xlabel("seconds from granule start")
axes[1, 0].set_ylabel("degrees")

axes[1, 1].scatter(ds["Cross_Track_Angle"].values, vza, c=seconds, s=3, cmap="plasma")
axes[1, 1].set_title("Viewing zenith vs cross-track angle")
axes[1, 1].set_xlabel("Cross_Track_Angle (deg)")
axes[1, 1].set_ylabel("Viewing_Zenith_Surface (deg)")

for ax in axes.flat[:3]:
    ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

summary = pd.DataFrame(
    {"min": [np.nanmin(v) for v in (vza, sza, vaz, saz, raa)],
     "max": [np.nanmax(v) for v in (vza, sza, vaz, saz, raa)],
     "range": [np.nanmax(v) - np.nanmin(v) for v in (vza, sza, vaz, saz, raa)]},
    index=["Viewing_Zenith", "Solar_Zenith", "Viewing_Azimuth", "Solar_Azimuth", "Relative_Azimuth"],
)
print(summary.round(3).to_string())

### 1.6 Cone and clock angles

`Cone_Angle` and `Clock_Angle` are the CERES SCI-12 pair: cone is the off-nadir angle, clock is the
azimuth about nadir measured in the inertial-velocity orbital frame. A cross-track scan therefore
appears in polar coordinates as two radial spokes near clock = 90 and clock = 270, with the
boresight running out to the limb and back along each.

`Clock_Angle_Rate` is the one field the product deliberately blanks. Clock angle is an azimuth about
nadir, so its derivative is singular there: as the scan crosses nadir the azimuth flips by roughly
180 degrees faster than 100 Hz sampling can resolve, making the finite difference an aliasing
artifact rather than a derivative. `libera_rad` fills the rate inside a nadir cone set by
`CLOCK_RATE_MIN_CONE_ANGLE_DEG`, which is a product decision rather than a curryer one.

Sizing that cone is not arbitrary. The largest surviving rate always sits at the gate boundary and
falls off as one over the gate squared: for a scan of angular speed `w` whose closest approach to
nadir is `b`, the rate at cone angle `g` is `w*b/g**2`. The gate is therefore chosen to bring the
surviving rate inside the field's declared `valid_range` with margin - the dotted lines in the right
panel mark that range.

In [ ]:
from libera_rad.constants import CLOCK_RATE_MIN_CONE_ANGLE_DEG

clock = ds["Clock_Angle"].values.astype("float64")
cone_rate = ds["Cone_Angle_Rate"].values.astype("float64")
clock_rate = ds["Clock_Angle_Rate"].values.astype("float64")
gate = float(CLOCK_RATE_MIN_CONE_ANGLE_DEG)

fig = plt.figure(figsize=(13, 5.5))

ax_polar = fig.add_subplot(1, 2, 1, projection="polar")
ax_polar.scatter(np.radians(clock), cone, c=seconds, s=3, cmap="plasma")
ax_polar.set_theta_zero_location("N")
ax_polar.set_theta_direction(-1)
ax_polar.set_rlim(0, 90)
ax_polar.set_title("Clock angle (azimuth) vs cone angle (radius)", pad=18)

ax_rate = fig.add_subplot(1, 2, 2)
ax_rate.plot(seconds, cone_rate, lw=0.8, label="Cone_Angle_Rate")
ax_rate.plot(seconds, clock_rate, lw=0.8, label="Clock_Angle_Rate")
ax_rate.fill_between(seconds, -80, 80, where=cone < gate, color="grey", alpha=0.3,
                     label=f"cone < {gate:g} deg (rate filled)")
declared_range = ds["Clock_Angle_Rate"].attrs.get("valid_range")
for edge in (float(declared_range[0]), float(declared_range[1])):
    ax_rate.axhline(edge, color="crimson", lw=1.0, ls=":")
ax_rate.plot([], [], color="crimson", lw=1.0, ls=":", label="Clock_Angle_Rate valid_range")
ax_rate.set_ylim(-80, 80)
ax_rate.set_xlabel("seconds from granule start")
ax_rate.set_ylabel("degrees / second")
ax_rate.set_title("Angle rates; shaded band is the near-nadir gate")
ax_rate.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

print("Clock_Angle range        : %.2f .. %.2f deg" % (np.nanmin(clock), np.nanmax(clock)))
print("samples inside the gate  : %d" % int(np.nansum(cone < gate)))
print("Clock_Angle_Rate NaN     : %d" % int(np.isnan(clock_rate).sum()))
print("Clock_Angle_Rate extremes: %.2f .. %.2f deg/s" % (np.nanmin(clock_rate), np.nanmax(clock_rate)))
print("declared valid_range     :", ds["Clock_Angle_Rate"].attrs.get("valid_range"))

### 1.7 Spacecraft state

`Satellite_Position` and `Satellite_Velocity` are J2000 (inertial), while the attitude quaternion
targets the Earth-fixed frame - a deliberate asymmetry in the product definition, and a common
source of confusion when combining them. `Satellite_Position_Start_Of_Hour` lives on a different
axis entirely: 24 epochs at the top of each UTC hour of the granule's day, not the sample grid.

In [ ]:
position = ds["Satellite_Position"].values
velocity = ds["Satellite_Velocity"].values
quaternion = np.stack([ds[f"Satellite_Attitude_Q{i}"].values for i in range(4)], axis=1)
hourly_position = ds["Satellite_Position_Start_Of_Hour"].values

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for i, comp in enumerate("XYZ"):
    axes[0, 0].plot(seconds, position[:, i], lw=1.0, label=comp)
axes[0, 0].set_title("Satellite_Position (J2000)")
axes[0, 0].set_ylabel("km")

for i, comp in enumerate("XYZ"):
    axes[0, 1].plot(seconds, velocity[:, i], lw=1.0, label="V" + comp)
axes[0, 1].set_title("Satellite_Velocity (J2000)")
axes[0, 1].set_ylabel("km/s")

for i in range(4):
    axes[1, 0].plot(seconds, quaternion[:, i], lw=1.0, label=f"Q{i}")
axes[1, 0].set_title("Satellite_Attitude quaternion (Earth-fixed target frame)")
axes[1, 0].set_xlabel("seconds from granule start")

axes[1, 1].plot(np.arange(24), np.linalg.norm(hourly_position, axis=1), "o-", lw=1.0,
                label="|Position_Start_Of_Hour|")
axes[1, 1].axhline(np.linalg.norm(position, axis=1).mean(), color="tab:red", ls="--",
                   label="mean in-granule radius")
axes[1, 1].set_title("Start-of-hour grid: 24 epochs, not the sample grid")
axes[1, 1].set_xlabel("hour of the granule's UTC day")
axes[1, 1].set_ylabel("km")

for ax in axes.flat:
    ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

print("orbital radius   : %.3f .. %.3f km" % (np.linalg.norm(position, axis=1).min(), np.linalg.norm(position, axis=1).max()))
print("speed            : %.4f .. %.4f km/s" % (np.linalg.norm(velocity, axis=1).min(), np.linalg.norm(velocity, axis=1).max()))
print("start-of-hour rows: %s, finite rows: %d" % (hourly_position.shape, int(np.isfinite(hourly_position).all(axis=1).sum())))

## 2. Consistency checks

Each check below is an identity or physical relation that must hold if the fields agree with one
another. They are the first pass on a new granule: they need no external truth data, only the
product itself, so they catch frame mix-ups, unit errors, stale fields and broken fill logic.

The fields are stored as float32, so exact identities bottom out around `1e-5` degrees. Two checks
are approximate by nature and are labelled as such: the spherical sine rule ignores Earth's
flattening, and the rate comparisons re-derive a derivative by finite difference.

In [ ]:
def central_difference(values: np.ndarray, times: np.ndarray) -> np.ndarray:
    """Central finite difference, NaN wherever the three-point stencil touches a NaN."""
    out = np.full_like(values, np.nan)
    finite = np.isfinite(values)
    usable = finite[:-2] & finite[1:-1] & finite[2:]
    out[1:-1] = np.where(usable, (values[2:] - values[:-2]) / (times[2:] - times[:-2]), np.nan)
    return out


def angular_residual(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Absolute difference between two angle arrays in degrees, wrapped to [0, 180]."""
    diff = np.abs(a - b) % 360.0
    return np.minimum(diff, 360.0 - diff)


checks = []


def record(name, residual, tolerance, unit, n, note=""):
    """Append one check result; `residual` is compared against `tolerance`."""
    checks.append(
        {
            "check": name,
            "residual": residual,
            "tolerance": tolerance,
            "unit": unit,
            "n": n,
            "verdict": "pass" if (np.isfinite(residual) and residual <= tolerance) else "REVIEW",
            "note": note,
        }
    )


def finite_max(a, b):
    """Max absolute difference over samples where both inputs are finite, with the count."""
    both = np.isfinite(a) & np.isfinite(b)
    return (np.max(np.abs(a[both] - b[both])) if both.any() else np.nan), int(both.sum())

In [ ]:
# Definitional identities within the product.
for lat_name, colat_name in [
    ("Latitude", "Colatitude"),
    ("Subsatellite_Latitude", "Subsatellite_Colatitude"),
    ("Subsolar_Latitude", "Subsolar_Colatitude"),
]:
    residual, n = finite_max(90.0 - ds[lat_name].values.astype("float64"), ds[colat_name].values.astype("float64"))
    record(f"{colat_name} == 90 - {lat_name}", residual, 1e-4, "deg", n)

los = ds["Line_Of_Sight"].values.astype("float64")
norm = np.linalg.norm(los, axis=1)
record("|Line_Of_Sight| == 1", np.nanmax(np.abs(norm - 1)), 1e-6, "-", int(np.isfinite(norm).sum()))

quat_norm = np.linalg.norm(quaternion.astype("float64"), axis=1)
record("|Satellite_Attitude| == 1", np.nanmax(np.abs(quat_norm - 1)), 1e-6, "-", int(np.isfinite(quat_norm).sum()))

residual, n = finite_max(radius, np.linalg.norm(position.astype("float64"), axis=1))
record("Radius == |Satellite_Position|", residual, 1e-6, "km", n)

# Relative azimuth is defined from the other two azimuths.
predicted_raa = np.mod(vaz.astype("float64") - saz.astype("float64") + 180.0, 360.0)
both = np.isfinite(predicted_raa) & np.isfinite(raa)
record("Relative_Azimuth == mod(viewing - solar + 180, 360)",
       float(angular_residual(predicted_raa[both], raa[both].astype("float64")).max()), 1e-3, "deg", int(both.sum()))

# Cone angle is the resultant of the two look angles.
along = np.radians(ds["Along_Track_Angle"].values.astype("float64"))
cross = np.radians(ds["Cross_Track_Angle"].values.astype("float64"))
predicted_cone = np.degrees(np.arctan(np.hypot(np.tan(along), np.tan(cross))))
residual, n = finite_max(predicted_cone, cone)
record("Cone_Angle == atan(hypot(tan(along), tan(cross)))", residual, 1e-3, "deg", n)

# Solar zenith must equal the great-circle angle from the surface point to the subsolar point.
phi, lam = np.radians(lat.astype("float64")), np.radians(lon.astype("float64"))
sun_phi = np.radians(ds["Subsolar_Latitude"].values.astype("float64"))
sun_lam = np.radians(ds["Subsolar_Longitude"].values.astype("float64"))
cos_sza = np.sin(phi) * np.sin(sun_phi) + np.cos(phi) * np.cos(sun_phi) * np.cos(lam - sun_lam)
residual, n = finite_max(np.degrees(np.arccos(np.clip(cos_sza, -1, 1))), sza.astype("float64"))
record("Solar_Zenith == great-circle(surface, subsolar)", residual, 1e-2, "deg", n)

# Spherical sine rule links cone angle at the spacecraft to viewing zenith at the surface.
a_axis, b_axis = 6378.1366, 6356.7519
local_radius = np.sqrt(
    ((a_axis**2 * np.cos(phi)) ** 2 + (b_axis**2 * np.sin(phi)) ** 2)
    / ((a_axis * np.cos(phi)) ** 2 + (b_axis * np.sin(phi)) ** 2)
)
predicted_vza = np.degrees(np.arcsin(np.clip(radius / local_radius * np.sin(np.radians(cone)), -1, 1)))
residual, n = finite_max(predicted_vza, vza.astype("float64"))
record("Viewing_Zenith == asin(Rsc/Re * sin(cone))", residual, 0.5, "deg", n,
       "approximate: spherical sine rule ignores flattening")

# Rates must reproduce a finite difference of their own field.
residual, n = finite_max(central_difference(cone, seconds), cone_rate)
record("Cone_Angle_Rate == d(Cone_Angle)/dt", residual, 5e-2, "deg/s", n, "approximate: finite difference")

outside_gate = cone >= gate
clock_fd = central_difference(clock, seconds)
both = np.isfinite(clock_fd) & np.isfinite(clock_rate) & outside_gate
record("Clock_Angle_Rate == d(Clock_Angle)/dt outside the gate",
       float(np.max(np.abs(clock_fd[both] - clock_rate[both]))), 5e-2, "deg/s", int(both.sum()),
       "approximate: finite difference")

# Velocity must reproduce the derivative of position.
position_fd = np.stack([central_difference(position[:, i].astype("float64"), seconds) for i in range(3)], axis=1)
usable = np.isfinite(position_fd).all(axis=1)
record("Satellite_Velocity == d(Satellite_Position)/dt",
       float(np.max(np.abs(position_fd[usable] - velocity[usable].astype("float64")))), 1e-3, "km/s",
       int(usable.sum()), "approximate: finite difference")

# Fill logic: the gate must be applied exactly, and surface masks must agree exactly.
in_gate = np.isfinite(cone) & (cone < gate)
record("Clock_Angle_Rate filled exactly inside the gate",
       float(np.count_nonzero(np.isfinite(clock_rate) & in_gate)), 0, "samples", int(in_gate.sum()))
record("surface fields share one NaN mask",
       float(sum(not np.array_equal(np.isnan(ds[f].values), reference_mask) for f in SURFACE_FIELDS)),
       0, "fields", len(SURFACE_FIELDS))

# Motor telemetry against derived pointing: the elevation encoder and the cross-track look angle
# describe the same motion in oppositely wound frames, offset by the fixed boresight rotation.
elevation = ds["Elevation"].values.astype("float64")
cross_track = ds["Cross_Track_Angle"].values.astype("float64")
paired = np.isfinite(elevation) & np.isfinite(cross_track)
boresight_offset = float(np.median(cross_track[paired] + elevation[paired]))
record("Cross_Track_Angle == -Elevation + constant offset",
       float(np.max(np.abs(cross_track[paired] + elevation[paired] - boresight_offset))), 5e-3, "deg",
       int(paired.sum()), f"fixed offset {boresight_offset:+.4f} deg from the FK boresight rotation")

# Declared valid_range conformance across every variable that declares one.
violations = {}
for name in ds.data_vars:
    declared = ds[name].attrs.get("valid_range")
    if declared is None:
        continue
    values = ds[name].values.astype("float64")
    finite = np.isfinite(values)
    if not finite.any():
        continue
    outside = finite & ((values < float(declared[0])) | (values > float(declared[1])))
    if outside.any():
        violations[name] = (int(outside.sum()), float(np.nanmin(values[finite])), float(np.nanmax(values[finite])),
                            float(declared[0]), float(declared[1]))
record("all fields inside declared valid_range", float(len(violations)), 0, "fields", len(ds.data_vars))

results = pd.DataFrame(checks)
results["residual"] = results["residual"].map(lambda v: f"{v:.4g}")
results["tolerance"] = results["tolerance"].map(lambda v: f"{v:g}")
print(results.to_string(index=False))

if violations:
    print("\nvalid_range violations:")
    for name, (count, lo, hi, dlo, dhi) in violations.items():
        print(f"  {name}: {count} samples outside [{dlo:g}, {dhi:g}]; actual range [{lo:.4g}, {hi:.4g}]")

### 2.1 What the checks turn up

Every check passes on this granule. The definitional identities close at float32 precision and the
fill logic is exact: the near-nadir gate is applied to precisely the samples inside it, and all nine
surface fields share one NaN mask. The motor-encoder check is worth singling out because it is the
only one that leaves the curryer-derived fields entirely: `Cross_Track_Angle` reproduces the negated
`Elevation` encoder to about a thousandth of a degree once a constant offset is removed, so the CK,
the FK boresight rotation and the derived look angle all agree.

A clean sweep is not the interesting case, though. The `valid_range` check earns its place because it
recently failed: with a 6-degree gate, `Clock_Angle_Rate` reached about +/- 43 deg/s against a
declared range of `[-20, 20]`. The gate did suppress the true singularity - ungated the rate
approaches 8000 deg/s - but not by enough. Because the surviving maximum scales as one over the gate
squared, widening the gate to 12 degrees brings it to roughly 10 deg/s, at the cost of filling 15% of
samples rather than 7.5%. See `CLOCK_RATE_MIN_CONE_ANGLE_DEG` in `libera_rad/constants.py`; the width
remains a placeholder pending science confirmation (`TODO[LIBSDC-739]`), since other scan modes
change both the scan speed and how close the boresight passes to nadir.

That is the class of problem these checks exist to surface: nothing raised an exception, the product
wrote cleanly, and write-time conformance checking still passed, because `valid_range` is declarative
metadata rather than an enforced constraint.

## 3. Reproducing the fields from SPICE

The geometry fields come from curryer's `GeometryData`, wrapped by `libera_rad.geolocation`. Two
calls cover the product: one against the spacecraft observer for state and reference ground points,
one against the instrument observer for everything boresight-derived. `calculate_geometry` makes
both and joins them.

Recomputing on the product's own timestamps and differencing against the file is the strongest
single check available - it validates the whole chain from kernels through packaging.

In [ ]:
from libera_utils.libera_spice.kernel_manager import KernelManager

kernel_sources = [str(p) for p in sorted(KERNEL_DIR.iterdir()) if p.suffix in (".bc", ".bsp")]
for source in kernel_sources:
    print(Path(source).name)

# Kernels already in the libera_utils cache warn about a basename collision on every load, on both
# the warnings and logging channels. Harmless here - the cached file is a copy of this same file -
# and the message is long enough to bury the rest of the notebook, so both channels are quieted.
kernel_log = logging.getLogger("libera_utils.libera_spice.kernel_manager")
previous_level = kernel_log.level

kernel_manager = KernelManager()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*basename conflict.*")
    kernel_log.setLevel(logging.ERROR)
    try:
        kernel_manager.load_libera_dynamic_kernels(kernel_sources, needs_naif_kernels=True, needs_static_kernels=True)
    finally:
        kernel_log.setLevel(previous_level)
kernel_manager.ensure_known_kernels_are_furnished()
print("\nkernels furnished")

In [ ]:
from libera_rad.geolocation import calculate_geometry, calculate_lat_lon_altitude

geometry = calculate_geometry(kernel_manager, time)
boresight_lla = calculate_lat_lon_altitude(kernel_manager, pd.DatetimeIndex(time))

print("curryer columns returned:", len(geometry.columns))
print(sorted(geometry.columns))

In [ ]:
# Product variable -> curryer column. Vector fields expand to three columns.
SCALAR_MAP = {
    "Subsatellite_Latitude": "subsatellite_latitude",
    "Subsatellite_Longitude": "subsatellite_longitude",
    "Subsatellite_Colatitude": "subsatellite_colatitude",
    "Subsolar_Latitude": "subsolar_latitude",
    "Subsolar_Longitude": "subsolar_longitude",
    "Subsolar_Colatitude": "subsolar_colatitude",
    "Radius_of_Satellite_from_Center_of_Earth": "spacecraft_radius",
    "Viewing_Zenith_Surface": "viewing_zenith",
    "Solar_Zenith_Surface": "solar_zenith",
    "Viewing_Azimuth_Surface_WRT_North": "viewing_azimuth",
    "Solar_Azimuth_Surface_WRT_North": "solar_azimuth",
    "Relative_Azimuth_Surface": "relative_azimuth",
    "Cone_Angle": "cone_angle",
    "Cone_Angle_Rate": "cone_angle_rate",
    "Clock_Angle": "clock_angle",
    "Along_Track_Angle": "along_track_angle",
    "Cross_Track_Angle": "cross_track_angle",
    "Satellite_Attitude_Q0": "attitude_q0",
    "Satellite_Attitude_Q1": "attitude_q1",
    "Satellite_Attitude_Q2": "attitude_q2",
    "Satellite_Attitude_Q3": "attitude_q3",
}
VECTOR_MAP = {
    "Satellite_Position": ["spacecraft_position_inertial_x", "spacecraft_position_inertial_y",
                           "spacecraft_position_inertial_z"],
    "Satellite_Velocity": ["spacecraft_velocity_inertial_x", "spacecraft_velocity_inertial_y",
                           "spacecraft_velocity_inertial_z"],
    "Line_Of_Sight": ["boresight_inertial_x", "boresight_inertial_y", "boresight_inertial_z"],
}

rows = []
for product_name, column in SCALAR_MAP.items():
    stored = ds[product_name].values.astype("float64")
    recomputed = geometry[column].to_numpy().astype("float64")
    residual, n = finite_max(stored, recomputed)
    rows.append({"product variable": product_name, "curryer column": column, "max |diff|": residual,
                 "n": n, "NaN masks match": bool(np.array_equal(np.isnan(stored), np.isnan(recomputed)))})

for product_name, columns in VECTOR_MAP.items():
    stored = ds[product_name].values.astype("float64")
    recomputed = geometry[columns].to_numpy().astype("float64")
    usable = np.isfinite(stored).all(axis=1) & np.isfinite(recomputed).all(axis=1)
    rows.append({"product variable": product_name, "curryer column": "(x, y, z)",
                 "max |diff|": float(np.max(np.abs(stored[usable] - recomputed[usable]))),
                 "n": int(usable.sum()),
                 "NaN masks match": bool(np.array_equal(np.isnan(stored).all(axis=1), np.isnan(recomputed).all(axis=1)))})

for product_name, column in [("Latitude", "lat"), ("Longitude", "lon"), ("Altitude", "alt")]:
    stored = ds[product_name].values.astype("float64")
    recomputed = boresight_lla[column].to_numpy().astype("float64")
    residual, n = finite_max(stored, recomputed)
    rows.append({"product variable": product_name, "curryer column": f"ellipsoid intersection [{column}]",
                 "max |diff|": residual, "n": n,
                 "NaN masks match": bool(np.array_equal(np.isnan(stored), np.isnan(recomputed)))})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nlargest disagreement anywhere: %.3g" % comparison["max |diff|"].max())
print("NaN masks match everywhere   :", bool(comparison["NaN masks match"].all()))

Every field reproduces to float32 rounding and every NaN mask matches, so the product carries
exactly what the library computes for these timestamps and kernels.

`Clock_Angle_Rate` is absent from the table above because it is the one geometry field the product
modifies after curryer returns it. Applying the gate reproduces the stored field exactly, which
locates that decision unambiguously in `libera_rad` rather than in curryer.

In [ ]:
raw_clock_rate = geometry["clock_angle_rate"].to_numpy().astype("float64")
gated_clock_rate = np.where(geometry["cone_angle"].to_numpy() < gate, np.nan, raw_clock_rate)

residual, n = finite_max(clock_rate, gated_clock_rate)
print("max |product - gated curryer| : %.4g deg/s over %d samples" % (residual, n))
print("NaN masks match               :", np.array_equal(np.isnan(clock_rate), np.isnan(gated_clock_rate)))
print("ungated |rate| maximum        : %.1f deg/s" % np.nanmax(np.abs(raw_clock_rate)))
print("gated   |rate| maximum        : %.1f deg/s" % np.nanmax(np.abs(gated_clock_rate)))
print("declared valid_range          :", ds["Clock_Angle_Rate"].attrs.get("valid_range"))

### 3.1 An independent path to the subsatellite point

The checks in section 2 stayed inside the product. This one leaves it: rotate the stored J2000
`Satellite_Position` into the Earth-fixed frame with SPICE and convert to geodetic coordinates. The
result must land on `Subsatellite_Latitude`/`Longitude`, which curryer computed by a different
route. Agreement here exercises the inertial-to-Earth-fixed transform that any consumer combining
the position field with ground coordinates has to get right.

In [ ]:
from curryer import spicetime
from curryer import spicierpy as sp
from curryer.compute import spatial

ephemeris_time = spicetime.adapt(np.asarray(time, dtype="datetime64[ns]"), "dt64", "et")
sampled = np.arange(0, len(time), 100)

earth_fixed = np.array([sp.pxform("J2000", "ITRF93", float(et)) @ position[i]
                        for i, et in zip(sampled, ephemeris_time[sampled])])
geodetic = spatial.ecef_to_geodetic(earth_fixed, degrees=True, meters=False)

print("samples compared            : %d" % len(sampled))
print("max |lat - Subsatellite_Lat|: %.3g deg" % np.max(np.abs(geodetic[:, 1] - sub_lat[sampled])))
print("max |lon - Subsatellite_Lon|: %.3g deg" % np.max(np.abs(geodetic[:, 0] - sub_lon[sampled])))
print("geodetic altitude           : %.3f .. %.3f km" % (geodetic[:, 2].min(), geodetic[:, 2].max()))

### 3.2 The radiometer field of view

The product reports one lat/lon per sample: the boresight intersection. The instrument actually
integrates over a finite cone, defined in the Libera IK (`LIBERA_RAD`, circular, 1.0 degree
half-angle). Projecting the FOV boundary instead of just the boresight shows the real ground
footprint, which is what footprint matching against the WFOV camera has to work with.

The footprint is near-circular at nadir and stretches rapidly toward the limb, since the same angular
cone subtends far more ground at grazing incidence. The left panel places the footprints along the
scan; the right panel re-centres each one and plots it in kilometres, which is the only way to
compare shapes that sit thousands of kilometres apart.

In [ ]:
fov_shape, fov_frame, boresight_vector, _, boundary = sp.getfov(sp.obj.Body("LIBERA_RAD").id, 10)
half_angle = np.degrees(np.arccos(boundary[0][2] / np.linalg.norm(boundary[0])))
print("IK FOV: %s in %s, boresight %s, half-angle %.4f deg" % (fov_shape, fov_frame, boresight_vector, half_angle))


def fov_boundary_vectors(half_angle_deg: float, n: int = 121) -> np.ndarray:
    """Unit vectors around a circular FOV boundary, in the instrument frame."""
    azimuth = np.linspace(0, 2 * np.pi, n)
    theta = np.radians(half_angle_deg)
    return np.stack(
        [np.sin(theta) * np.cos(azimuth), np.sin(theta) * np.sin(azimuth), np.full_like(azimuth, np.cos(theta))],
        axis=1,
    )


# Sample a spread of cone angles that are all on the Earth, so every footprint closes.
candidates = np.where(np.isfinite(lat) & (cone < 60))[0]
chosen = candidates[np.argsort(cone[candidates])][np.linspace(0, len(candidates) - 1, 6).astype(int)]

chosen_ugps = spicetime.adapt(pd.DatetimeIndex(time[chosen]), "iso")
footprints, _, _ = spatial.compute_ellipsoid_intersection(
    chosen_ugps,
    sp.obj.Body("LIBERA_RAD", frame=True),
    custom_pointing_vectors=fov_boundary_vectors(half_angle),
    give_geodetic_output=True,
    give_lat_lon_in_degrees=True,
)

# groupby returns groups in uGPS order, which is not the cone-angle order of `chosen`;
# map each group back to its sample index explicitly rather than relying on position.
sample_of_ugps = dict(zip(chosen_ugps, chosen))

colors = dict(zip(chosen, plt.cm.viridis(np.linspace(0, 0.85, len(chosen)))))

fig = plt.figure(figsize=(14, 7))
ax_map = fig.add_subplot(1, 2, 1, projection=ccrs.Orthographic(float(np.nanmean(lon)), float(np.nanmean(lat))))
ax_map.set_global()
ax_map.coastlines(linewidth=0.6)
ax_map.gridlines(linewidth=0.3, linestyle="--", alpha=0.6)
ax_map.plot(lon[np.isfinite(lat)], lat[np.isfinite(lat)], color="grey", lw=0.6, alpha=0.7,
            transform=ccrs.PlateCarree(), zorder=5, label="boresight track")
ax_local = fig.add_subplot(1, 2, 2)

sizes = []
for ugps_key, group in footprints.groupby(level=0):
    index = sample_of_ugps[ugps_key]
    ring_lat, ring_lon = group["lat"].to_numpy(), group["lon"].to_numpy()
    if not np.isfinite(ring_lat).all():
        continue
    label = "cone %.1f deg" % cone[index]
    ax_map.plot(np.append(ring_lon, ring_lon[0]), np.append(ring_lat, ring_lat[0]), color=colors[index],
                lw=1.8, transform=ccrs.PlateCarree(), zorder=8)
    ax_map.scatter(ring_lon.mean(), ring_lat.mean(), color=colors[index], s=18,
                   transform=ccrs.PlateCarree(), zorder=9, label=label)

    # Re-centre each ring on its own centre and convert to kilometres, so shape and scale are
    # comparable between footprints that sit thousands of kilometres apart.
    centre_lat, centre_lon = ring_lat.mean(), ring_lon.mean()
    east_km = (ring_lon - centre_lon) * 111.32 * np.cos(np.radians(centre_lat))
    north_km = (ring_lat - centre_lat) * 111.32
    ax_local.plot(np.append(east_km, east_km[0]), np.append(north_km, north_km[0]),
                  color=colors[index], lw=1.8, label=label)
    sizes.append({"cone angle (deg)": round(float(cone[index]), 2),
                  "viewing zenith (deg)": round(float(vza[index]), 2),
                  "N-S extent (km)": round(float(north_km.max() - north_km.min()), 1),
                  "E-W extent (km)": round(float(east_km.max() - east_km.min()), 1)})

handles, labels = ax_map.get_legend_handles_labels()
order = sorted(range(len(labels)), key=lambda i: (labels[i] == "boresight track", labels[i]))
ax_map.legend([handles[i] for i in order], [labels[i] for i in order], loc="lower left", fontsize=9)
ax_map.set_title("Footprint locations along the scan")

ax_local.set_aspect("equal")
ax_local.axhline(0, color="grey", lw=0.5)
ax_local.axvline(0, color="grey", lw=0.5)
ax_local.set_xlabel("east offset from footprint centre (km)")
ax_local.set_ylabel("north offset (km)")
ax_local.set_title("Same footprints, each re-centred (km)")
local_handles, local_labels = ax_local.get_legend_handles_labels()
local_order = sorted(range(len(local_labels)), key=lambda i: local_labels[i])
ax_local.legend([local_handles[i] for i in local_order], [local_labels[i] for i in local_order],
                fontsize=9, loc="upper right")

fig.suptitle("Radiometer FOV footprint (%.1f deg half-angle) across the scan" % half_angle, fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

print(pd.DataFrame(sorted(sizes, key=lambda row: row["cone angle (deg)"])).to_string(index=False))

### 3.3 Any time grid

`calculate_geometry` takes arbitrary timestamps, so it is not limited to a product's sample grid.
The in-repo kernels cover a far longer window than the 30-second granule: the AZROT and ELSCAN CKs
run to 19:05:49. Recomputing at 10 Hz across the full window covers most of an orbit and shows the
along-track behaviour the short granule cannot.

This is also the pattern for computing geometry on any other input - a different granule, a
candidate kernel set, or a proposed scan pattern.

In [ ]:
import time as timing

# Window covered by the in-repo AZROT-CK / ELSCAN-CK for this granule.
long_grid = pd.date_range("2025-11-20T18:00:00", "2025-11-20T19:05:00", freq="100ms")

started = timing.perf_counter()
long_geometry = calculate_geometry(kernel_manager, long_grid.values)
elapsed = timing.perf_counter() - started
print("%d samples (%.1f min at 10 Hz) in %.1f s" % (len(long_grid), len(long_grid) / 600, elapsed))

long_seconds = (long_grid - long_grid[0]).total_seconds().to_numpy()
long_sub_lat = long_geometry["subsatellite_latitude"].to_numpy()
long_sub_lon = long_geometry["subsatellite_longitude"].to_numpy()
long_vza = long_geometry["viewing_zenith"].to_numpy()

fig = plt.figure(figsize=(14, 5.5))

ax_map = fig.add_subplot(1, 3, 1, projection=ccrs.Robinson())
ax_map.set_global()
ax_map.coastlines(linewidth=0.5)
ax_map.gridlines(linewidth=0.3, linestyle="--", alpha=0.5)
ax_map.scatter(long_sub_lon, long_sub_lat, s=1, c=long_seconds / 60, cmap="plasma",
               transform=ccrs.PlateCarree(), zorder=6)
ax_map.set_title("Subsatellite track,\nfull kernel window")

# The scan sawtooth has a ~5 s period, so plotting 39,000 samples of it fills a solid band. The
# orbit-scale panel shows the slow quantities; the zoom panel shows the scan still running.
ax_slow = fig.add_subplot(1, 3, 2)
ax_slow.plot(long_seconds / 60, long_sub_lat, lw=1.2, label="subsatellite latitude")

# Solar zenith at the boresight swings by tens of degrees within every 5 s sweep, so at orbit scale
# it is a band rather than a curve. The great-circle angle to the subsolar point evaluated at the
# subsatellite point is the smooth, scan-independent version of the same quantity.
sub_phi, sub_lam = np.radians(long_sub_lat), np.radians(long_sub_lon)
sun_phi_long = np.radians(long_geometry["subsolar_latitude"].to_numpy())
sun_lam_long = np.radians(long_geometry["subsolar_longitude"].to_numpy())
nadir_sza = np.degrees(np.arccos(np.clip(
    np.sin(sub_phi) * np.sin(sun_phi_long)
    + np.cos(sub_phi) * np.cos(sun_phi_long) * np.cos(sub_lam - sun_lam_long), -1, 1)))

ax_slow.plot(long_seconds / 60, long_geometry["solar_zenith"].to_numpy(), lw=0.5, color="tab:orange",
             alpha=0.25, label="solar zenith at boresight (scan spread)")
ax_slow.plot(long_seconds / 60, nadir_sza, lw=1.4, color="tab:red", label="solar zenith at nadir point")
ax_slow.axhline(90, color="grey", lw=0.8, ls="--", label="terminator (90 deg)")
ax_slow.set_xlabel("minutes from 18:00:00")
ax_slow.set_ylabel("degrees")
ax_slow.set_title("Orbit-scale geometry")
ax_slow.legend(loc="lower left", fontsize=8)

zoom = long_seconds <= 20
ax_zoom = fig.add_subplot(1, 3, 3)
ax_zoom.plot(long_seconds[zoom], long_geometry["cone_angle"].to_numpy()[zoom], lw=1.0, color="tab:green",
             label="cone_angle")
ax_zoom.set_xlabel("seconds from 18:00:00")
ax_zoom.set_ylabel("degrees")
ax_zoom.set_title("First 20 s: the scan\nis unchanged")
ax_zoom.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

print("subsatellite latitude : %.2f .. %.2f deg" % (long_sub_lat.min(), long_sub_lat.max()))
print("boresight on Earth    : %d of %d samples (%.1f%%)" % (
    np.isfinite(long_vza).sum(), len(long_vza), 100 * np.isfinite(long_vza).mean()))
print("cone angle            : %.2f .. %.2f deg" % (
    np.nanmin(long_geometry["cone_angle"]), np.nanmax(long_geometry["cone_angle"])))

### 3.4 Degraded modes

Two manifest configurations produce geometry without full pointing, and it is worth knowing what
they look like so a degraded granule is not mistaken for a broken one.

- **`jpss_only`** - no motor CK, so the instrument frame does not resolve and every boresight field
  is NaN. The spacecraft fields still come through, and geolocation falls back to the subsatellite
  point.
- **`use_geo: false`** - geolocation is bypassed entirely and every geometry field is written as its
  product fill value (`-999` for angles, `-9999` for distances). Note these are fill values, not
  NaN, so anything reading the file without `mask_and_scale` sees the sentinels directly.

In [ ]:
from libera_rad.geolocation import create_placeholder_geometry, subsatellite_lat_lon_alt

fallback = subsatellite_lat_lon_alt(geometry)
print("jpss_only geolocation falls back to the subsatellite point:")
print(fallback.head(3).to_string(), "\n")
print("max |fallback lat - Subsatellite_Latitude| : %.3g deg"
      % np.max(np.abs(fallback["lat"].to_numpy() - sub_lat)))
print("fallback altitude is the spacecraft altitude: %.3f .. %.3f km\n"
      % (fallback["alt"].min(), fallback["alt"].max()))

placeholder = create_placeholder_geometry(len(time))
distinct = {column: float(placeholder[column].iloc[0]) for column in placeholder.columns}
print("use_geo: false writes constant fill values across %d columns:" % len(placeholder.columns))
for value in sorted(set(distinct.values())):
    members = [name for name, fill in distinct.items() if fill == value]
    print("  %8.1f  <- %d columns (e.g. %s)" % (value, len(members), ", ".join(members[:3])))

## 4. Checklist for a new granule

Condensed from section 2. None of these need external truth data:

1. **Coverage.** Do the surface fields share one NaN mask? Is the cut-off at the geometric limb
   `asin(Re/Rsc)` rather than a contiguous block in time (which would indicate a kernel gap)?
2. **Definitional identities.** `Colatitude == 90 - Latitude`, `|Line_Of_Sight| == 1`,
   `|quaternion| == 1`, `Radius == |Satellite_Position|`.
3. **Angle closure.** `Relative_Azimuth == mod(viewing - solar + 180, 360)`;
   `Cone_Angle == atan(hypot(tan(along), tan(cross)))`.
4. **Independent geometry.** Does `Solar_Zenith_Surface` match the great-circle angle from the
   surface point to the subsolar point? Does the J2000 position, rotated to Earth-fixed, land on the
   subsatellite point?
5. **Motor telemetry vs derived pointing.** Does `Cross_Track_Angle` track the negated `Elevation`
   encoder to within a constant offset? This is the one check that spans the CK, the FK boresight
   rotation and the geometry fields at once.
6. **Rates.** Do `Cone_Angle_Rate` and `Clock_Angle_Rate` reproduce finite differences of their own
   fields, and does `Satellite_Velocity` reproduce the derivative of `Satellite_Position`?
7. **Fill logic.** Is `Clock_Angle_Rate` blanked on exactly the samples inside the near-nadir gate?
8. **Declared ranges.** Is every field inside its own `valid_range`? This is metadata, not an
   enforced constraint, so a conforming write proves nothing here.
9. **Reproducibility.** Does `calculate_geometry` on the product's own timestamps and kernels
   reproduce every field to float32 rounding, with matching NaN masks?

Where to look next in the code:

| Topic | Location |
| --- | --- |
| Field selection, observer split, gate | `libera_rad/geolocation.py`, `libera_rad/constants.py` |
| Packaging into product variables | `libera_rad/l1b.py` (`_package_l1b_product`) |
| Declared units, ranges and fills | `libera_rad/data/L1B_RAD-4CH_product_definition.yml` |
| Field definitions and conventions | `curryer.compute.geometry` module docstring |
| Kernels and frames | `libera_utils/data/spice/jpss4/` (FK, IK) |